# Lab 01 — Python Refresher for Data People
**Junior Analyst Track** · Beginner · ~45 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Map Excel columns/rows/sheets to Python lists, dicts, and lists-of-dicts
2. Clean messy text and numbers with strip/title and None handling
3. Build reusable cleaning functions and comprehensions
4. Answer revenue-by-region from a 12-row messy sales sheet

## Datasets (this folder)
- `sales_spreadsheet.csv` — **upload** in Colab (or keep next to the notebook locally)

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-01-python-refresher-for-data-people/lab-01-python-refresher-for-data-people.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`sales_spreadsheet.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-01-python-refresher-for-data-people"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/labs/lab-01-python-refresher-for-data-people/bundle/dataset.zip"
NEED = ["sales_spreadsheet.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Junior Analyst Track: Cleaning a Sales Spreadsheet

> **Scenario:** You're a junior analyst. Someone sends you `sales_spreadsheet.csv` — 12 rows, messy product names, missing quantities, inconsistent regions. Your job: clean it with pure Python and answer: *what's our total revenue by region?*
>
> **You will learn:** lists, dicts, loops, functions, comprehensions — all with sales-data examples.
> **Time:** ~45 minutes. **Level:** Beginner. **Needs:** Python 3.8+ only (no pandas needed).

### Excel → Python mental map

| Excel idea | Python idea | Example |
|---|---|---|
| A column (e.g. all Prices) | a `list` | `[899.99, 25.5, 75.0]` |
| One row (e.g. Order 101) | a `dict` | `{"OrderID": 101, "Product": "Laptop"}` |
| Whole sheet | a `list of dicts` | `[{"OrderID":101, ...}, {"OrderID":102, ...}]` |

Keep that map in mind — everything below follows it.

### Setup: load the real dataset

Cell 0 already downloaded + unzipped `sales_spreadsheet.csv`. Load it below with the `csv` module — blanks become `None`, numbers become `int`/`float`. From here on, `sales` **is** the dataset file, and every exercise runs against it.

In [ ]:
# Load the real dataset file (Cell 0 fetched it via wget + unzip)
import csv

def _int_or_none(v):
    v = v.strip()
    return int(v) if v else None

def _float_or_none(v):
    v = v.strip()
    return float(v) if v else None

with open("sales_spreadsheet.csv", newline="", encoding="utf-8") as f:
    sales = [
        {
            "OrderID": int(r["OrderID"]),
            "Product": r["Product"],
            "Quantity": _int_or_none(r["Quantity"]),
            "Price": _float_or_none(r["Price"]),
            "Region": r["Region"],
        }
        for r in csv.DictReader(f)
    ]

print(len(sales), "rows from sales_spreadsheet.csv")  # 12
print(sales[0])
# {'OrderID': 101, 'Product': '  Laptop ', 'Quantity': 2, 'Price': 899.99, 'Region': 'West'}


Messy on purpose. Notice: extra spaces, ALL CAPS, empty string `""`, `None` for missing, `west` vs `West` vs `WEST`.

---

## 1. Lists — your columns

A list is an ordered collection. Think: one Excel column.

In [ ]:
prices = [899.99, 25.50, 75.00, 12.99]
products = ["Laptop", "Mouse", "Keyboard"]

# Indexing (0-based!) and slicing
print(prices[0])      # 899.99 — first item
print(prices[-1])     # 12.99 — last item
print(products[0:2])  # ['Laptop', 'Mouse'] — slice, end is exclusive

# Useful list ops for analysts
print(len(prices))    # 4 — row count
print(sum(prices))    # 1013.48 — like =SUM()
print(min(prices), max(prices))  # 12.99 899.99 — like =MIN(), =MAX()
print(sorted(products))  # alphabetical copy


Mutating a list (cleaning in place):

In [ ]:
raw_products = ["  Laptop ", "KEYBOARD", "mouse", ""]
cleaned = []
for p in raw_products:
    cleaned.append(p.strip())  # strip() removes spaces
print(cleaned)  # ['Laptop', 'KEYBOARD', 'mouse', '']

# Remove empties, normalise case
cleaned = [p.strip().title() for p in cleaned if p.strip() != ""]
print(cleaned)  # ['Laptop', 'Keyboard', 'Mouse']
# .title() -> 'KEYBOARD' becomes 'Keyboard'. Perfect for product names.


**Key methods to memorise:** `.append(x)`, `.remove(x)`, `len()`, `sum()`, `sorted()`, `list.count(x)`.

> Try it: what does `prices + [49.99]` do? (Answer: returns a new longer list, original unchanged.)

---

## 2. Dicts — your rows

A dict is labelled fields. Think: one spreadsheet row where headers are keys.

In [ ]:
order = {"OrderID": 101, "Product": "  Laptop ", "Quantity": 2, "Price": 899.99, "Region": "West"}

print(order["Product"])          # '  Laptop ' — direct access (errors if key missing)
print(order.get("Discount", 0))  # 0 — safe access with default, never crashes

order["Product"] = order["Product"].strip().title()  # clean that cell
order["Total"] = order["Quantity"] * order["Price"]  # add a calculated column
print(order)
# {'OrderID': 101, 'Product': 'Laptop', 'Quantity': 2, 'Price': 899.99, 'Region': 'West', 'Total': 1799.98}

print(list(order.keys()))    # headers
print(list(order.values()))  # row values
print(list(order.items()))   # pairs — you'll loop over these a lot


A whole sheet = list of dicts (you already have `sales`):

In [ ]:
print(sales[0])              # first row (Order 101)
print(sales[0]["Region"])    # 'West'
print(len(sales))            # 12 rows

# Like =VLOOKUP: find Order 108
order_108 = None
for row in sales:
    if row["OrderID"] == 108:
        order_108 = row
        break
print(order_108)


---

## 3. Loops — your cleaning engine

### 3a. `for` loop (you'll use this 95% of the time)

In [ ]:
# Pattern 1: total revenue, skipping bad rows
total = 0
for row in sales:
    qty = row["Quantity"]
    price = row["Price"]
    if qty is None or price is None:
        continue  # skip incomplete rows, like filtering blanks in Excel
    total += qty * price
print(f"Total (skip missing): ${total:.2f}")

# Pattern 2: enumerate when you need row numbers
for i, row in enumerate(sales, start=2):  # start=2 mimics Excel (header is row 1)
    if not str(row["Product"]).strip():
        print(f"Row {i}: missing product (Order {row['OrderID']})")

# Pattern 3: counting by group (like a PivotTable COUNTIF)
region_counts = {}
for row in sales:
    region = str(row["Region"]).strip().title()  # 'west' -> 'West', ' WEST' -> 'West'
    region_counts[region] = region_counts.get(region, 0) + 1
print(region_counts)  # {'West': 4, 'East': 4, 'North': 2, 'South': 1}


### 3b. `if / elif / else` inside loops

In [ ]:
for row in sales:
    raw = str(row["Product"]).strip()
    if raw == "":
        status = "NEEDS PRODUCT"
    elif row["Quantity"] is None or row["Price"] is None:
        status = "NEEDS NUMBERS"
    else:
        status = "OK"
    print(row["OrderID"], status)


> Rule of thumb: `strip()` text first, *then* compare. `" west " == "west"` is `False` — a classic junior bug.

---

## 4. Functions — reusable cleaning steps

If you copy-paste cleaning logic twice, make it a function.

In [ ]:
def clean_text(value):
    """Strip spaces and title-case. Empty/None -> ''."""
    if value is None:
        return ""
    return str(value).strip().title()


def clean_row(row):
    """Return a cleaned copy of a sales row + computed Total. Never mutates input."""
    cleaned = {
        "OrderID": row["OrderID"],
        "Product": clean_text(row["Product"]),
        "Quantity": row["Quantity"] if row["Quantity"] is not None else 0,
        "Price": row["Price"] if row["Price"] is not None else 0.0,
        "Region": clean_text(row["Region"]),
    }
    cleaned["Total"] = round(cleaned["Quantity"] * cleaned["Price"], 2)
    return cleaned


def revenue_by_region(rows):
    """Sum Total per Region. Expects already-cleaned rows."""
    totals = {}
    for r in rows:
        totals[r["Region"]] = totals.get(r["Region"], 0) + r["Total"]
    return totals


# Use them:
clean_sales = [clean_row(r) for r in sales]  # preview of next section!
print(clean_sales[0])
# {'OrderID': 101, 'Product': 'Laptop', 'Quantity': 2, 'Price': 899.99, 'Region': 'West', 'Total': 1799.98}

print(revenue_by_region(clean_sales))


Why functions help analysts:
- `clean_text` fixes *every* text column the same way.
- `clean_row` is testable: feed one row, check output.
- Default pattern `x if x is not None else 0` safely fills blanks (like Excel `=IF(ISBLANK(A2),0,A2)`).

---

## 5. Comprehensions — one-line transforms

Comprehensions are compact loops for building lists/dicts. Analysts love them for column ops.

In [ ]:
# List comprehension: [expression for item in iterable if condition]

# All totals in one line
totals = [r["Quantity"] * r["Price"] for r in sales
          if r["Quantity"] is not None and r["Price"] is not None]
print(totals)

# Clean product list, dropping blanks — compare to the loop in Section 1
products_clean = [r["Product"].strip().title() for r in sales if str(r["Product"]).strip()]
print(products_clean)

# Dict comprehension: {key_expr: value_expr for item in iterable}
# Revenue per order id
revenue_per_order = {r["OrderID"]: (r["Quantity"] or 0) * (r["Price"] or 0) for r in sales}
print(revenue_per_order[101])  # 1799.98
# (r["Quantity"] or 0) trick: None, 0, '' all become 0. Handy for messy sheets.

# Full clean pipeline in 2 lines (uses functions from Section 4)
clean_sales = [clean_row(r) for r in sales]
valid_sales = [r for r in clean_sales if r["Product"] != "" and r["Total"] > 0]
print(f"{len(valid_sales)} valid orders out of {len(sales)}")
print(revenue_by_region(valid_sales))


When *not* to use a comprehension: if the logic needs 3+ `if`s or side effects (printing, appending elsewhere) — use a plain loop for readability.

---

## Putting it together

In [ ]:
def analyse(sales_rows):
    cleaned = [clean_row(r) for r in sales_rows]
    valid = [r for r in cleaned if r["Product"] and r["Total"] > 0]
    return {
        "rows_in": len(sales_rows),
        "rows_valid": len(valid),
        "rows_dropped": len(sales_rows) - len(valid),
        "total_revenue": round(sum(r["Total"] for r in valid), 2),
        "by_region": revenue_by_region(valid),
    }

import json
print(json.dumps(analyse(sales), indent=2))
# Expect ~ total 4451.83, West highest. Run it!


---

## Exercises (do these!)

### Exercise 1 — High-value orders (lists + loops)
Using `sales` (raw), print `OrderID` and `Total` for every order where `Quantity` and `Price` exist and `Total > 500`. Also print how many such orders exist.
*Expected: 2 orders (101 and 106). Hint: `if qty is not None and price is not None:` then compute.*

**Follow-up:** What share of valid revenue comes from those high-value orders? Check: about 0.7184 (two orders drive ~72%).

<details>
<summary>Hint</summary>

Loop, compute `qty * price`, collect matches in a list, then print.
</details>

### Exercise 2 — Category counter (dicts + functions)
Write a function `category(product)` that returns `"Computer"` for Laptop/Monitor/Keyboard/Mouse (any case/spaces), `"Audio"` for Headset, else `"Other"` (and `"Unknown"` for blank). Then loop over `sales`, clean each product with `clean_text`, categorise it, and build a dict `{category: count}`.
*Expected counts approx: Computer 10, Audio 1, Unknown 1. Hint: reuse `clean_text` from Section 4.*

**Follow-up:** What share of the 12 orders are Computers? Check: 10/12, about 0.8333.

<details>
<summary>Hint</summary>

```python
def category(product):
    p = clean_text(product)
    ...
```

Use a dict + `.get(cat, 0) + 1` to count.
</details>

### Exercise 3 — One-line clean (comprehensions)
In **one** list comprehension (plus `clean_row` if you want), build `east_big` = list of cleaned rows where `Region == "East"` and `Total >= 100`. Then in one dict comprehension build `{OrderID: Total}` from it.
*Expected: Orders 102 (127.5), 108 (150.0), and 110 (199.96) — note 110’s Region is `"east"`, which `clean_text` normalises to `"East"`. Hint: `[clean_row(r) for r in sales if ...]` — but filter on cleaned values, so either clean twice or clean first then filter.*

**Follow-up:** Confirm east_big sums to East's regional total. Check: 477.46.

<details>
<summary>Hint</summary>

Easiest readable answer is two steps: `cleaned = [clean_row(r) for r in sales]` then `east_big = [r for r in cleaned if ...]`. One-liner is possible but two lines is more Pythonic — say so in a comment.
</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
big = []
for r in sales:
    qty, price = r["Quantity"], r["Price"]
    if qty is None or price is None:
        continue
    total = qty * price
    if total > 500:
        big.append((r["OrderID"], round(total, 2)))
        print(r["OrderID"], round(total, 2))
print("count:", len(big))  # 101 1799.98 / 106 899.99 / count: 2

# --- Solution 2 ---
def category(product):
    p = clean_text(product)
    if p == "":
        return "Unknown"
    if p in ("Laptop", "Monitor", "Keyboard", "Mouse"):
        return "Computer"
    if p == "Headset":
        return "Audio"
    return "Other"

counts = {}
for r in sales:
    cat = category(r["Product"])
    counts[cat] = counts.get(cat, 0) + 1
print(counts)  # {'Computer': 10, 'Unknown': 1, 'Audio': 1}

# --- Solution 3 ---
cleaned = [clean_row(r) for r in sales]
east_big = [r for r in cleaned if r["Region"] == "East" and r["Total"] >= 100]
east_map = {r["OrderID"]: r["Total"] for r in east_big}
print(east_big)
print(east_map)  # {102: 127.5, 108: 150.0, 110: 199.96}

# --- Follow-up 1 ---
valid = [r for r in sales if str(r["Product"]).strip()
         and r["Quantity"] is not None and r["Price"] is not None]
valid_rev = sum(r["Quantity"] * r["Price"] for r in valid)
share = round(sum(t for _, t in big) / valid_rev, 4)
print(share)  # ~0.7184 — two orders drive ~72% of valid revenue
assert abs(share - 0.7184) < 0.001

# --- Follow-up 2 ---
cshare = round(counts["Computer"] / len(sales), 4)
print(cshare)  # 0.8333
assert cshare == 0.8333

# --- Follow-up 3 ---
etot = round(sum(east_map.values()), 2)
print(etot)  # 477.46 == East regional total
assert etot == 477.46


### What to learn next
- `csv.DictReader` / `csv.DictWriter` to load/save the real file without pandas.
- `pathlib`, `datetime` for file dates.
- Then pandas: `DataFrame` = supercharged list-of-dicts.
- Cheat sheet: `list` → column, `dict` → row, `for` → row-by-row, `def` → reusable step, `[... for ... if ...]` → one-line transform.

*Files in this folder: `sales_spreadsheet.csv` (messy input) + this notebook in Markdown. Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, 
or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
